# Tokenization grid on a real building

**Exploratory / figure notebook** -- throwaway plotting code, exempt from the TDD and
reproducibility rules.

Shows the coordinate lattice the mesh tokenizer quantizes onto, overlaid on one LOD2
building from `mini` (`data/The Hague/mini_remote`). Every vertex is snapped to the
nearest lattice point; its three bin indices are its three tokens.

The frame is exactly the one `MeshDataset` uses: the **LOD1_synth** bounding box,
longest side = full range, `margin_hi = 0.1` on +z only (`MeshDataConfig` defaults).
So the quantization box is a cube of side `base` in x/y and `1.1 * base` in z, and the
building fills it only along its longest horizontal axis.

Rotate the figure to the angle you want, then use the camera icon in the plot's
toolbar -- it exports *the current view* at 3x resolution.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.dataset.mesh_dataset import (
    parse_cityjson_file_to_meshes, normalize_to_unit_box, quantize, dequantize, canonicalize,
    fix_winding,
)
from src.utils.config import MeshDataConfig

## Config

Chosen by ranking mini's 20k pairs for roof complexity (distinct roof-plane normals) at a
moderate face count, with LOD2 rising above the LOD1 box but inside the +z margin.
Alternatives from the same ranking, if you want a different look:

| object | tile | tris | note |
|---|---|---|---|
| `NL.IMBAG.Pand.1901100000024415-0` | `Source B/9-384-596.city.json` | 140 | default -- low-rise, stepped multi-part roof, no merges |
| `NL.IMBAG.Pand.0518100000346408-0` | `Source B/10-270-594.city.json` | 92 | tall corner block, 11 roof planes |
| `bag_0518100000238305` | `Source A/TheHague3D_batch_06_Wijk 21 Regentessekwartier_2022-06-21.json` | 62 | simpler, ridge +8% over LOD1 |
| `NL.IMBAG.Pand.0518100000334381-0` | `Source B/10-262-588.city.json` | 158 | busier roof |
| `NL.IMBAG.Pand.0484100000001408-0` | `Source B/8-360-600.city.json` | 88 | steep hip roof; 7 vertices merge, 10 faces collapse (atypical) |

In [ ]:
DATASET = ROOT / "data" / "The Hague" / "mini_remote"   # what the configs call `mini`
TILE    = "Source B/9-384-596.city.json"
OBJ_ID  = "NL.IMBAG.Pand.1901100000024415-0"

cfg       = MeshDataConfig()
NUM_BINS  = cfg.num_bins            # 128
MARGIN_LO = cfg.margin_lo           # [0, 0, 0]
MARGIN_HI = cfg.margin_hi           # [0, 0, 0.1]

# Full-resolution 3D lattice around every tokenized vertex: a cube LOCAL_N bins on a side,
# centred on the vertex's lattice point (n=4 -> lines at c-2..c+2, i.e. 4^3 cells).
LOCAL_N = 4
MESH_OPACITY = 0.45                 # < 1 so the half of each cube inside the building shows

MAJOR_EVERY = 16                    # darker line every N bins; 0 disables
# Grid planes drawn on the box faces. Pick the faces *behind* the building for your camera.
GRID_PLANES = ("z_min", "x_min", "y_max")
CAMERA = dict(eye=dict(x=1.35, y=-1.45, z=1.25), up=dict(x=0, y=0, z=1))
HEIGHT = 850

COLORS = dict(
    grid_minor="#e4e6ea", grid_major="#b4b9c2", qbox="#4a4f57", lod1box="#2a9d8f",
    mesh="#a9c9e6", edges="#1d4f86", vert="#1b1b1b", snapped="#e8710a", local="#8a919e",
)

## Load, normalize, quantize

In [ ]:
v1, f1 = parse_cityjson_file_to_meshes(DATASET / "LOD1_synth" / TILE)[OBJ_ID]
v2, f2 = parse_cityjson_file_to_meshes(DATASET / "LOD2" / TILE)[OBJ_ID]

# Both LODs in the LOD1 frame -- exactly the training-time normalization.
v2_n, center, scale = normalize_to_unit_box(v2, ref=v1, margin_lo=MARGIN_LO, margin_hi=MARGIN_HI)
q = quantize(v2_n, NUM_BINS)                      # [V, 3] bin indices = the tokens
v2_q = dequantize(q, NUM_BINS)

# Plot in local metres: origin at the LOD1 footprint's min corner, z = 0 at the ground.
origin = v1.min(axis=0)
to_m = lambda n: n * scale + center - origin      # normalized -> local metres

V_m, Vq_m = v2 - origin, to_m(v2_q)
step = scale / (NUM_BINS - 1)                     # lattice spacing per axis, metres
box_lo, box_hi = to_m(np.full(3, -0.5)), to_m(np.full(3, 0.5))
l1_lo, l1_hi = v1.min(0) - origin, v1.max(0) - origin

qv, qf = canonicalize(q, f2)
err = np.linalg.norm(Vq_m - V_m, axis=1)
print(f"{OBJ_ID}: {len(v2)} verts / {len(f2)} tris")
print(f"bin size  x {step[0]*100:.1f} cm   y {step[1]*100:.1f} cm   z {step[2]*100:.1f} cm")
print(f"snap error  mean {err.mean()*100:.1f} cm   max {err.max()*100:.1f} cm")
print(f"after quantization: {len(qv)} verts (merged {len(v2) - len(qv)}), "
      f"{len(qf)} tris (collapsed/duplicate {len(f2) - len(qf)})")
print(f"used bins  x {q[:,0].min()}-{q[:,0].max()}   y {q[:,1].min()}-{q[:,1].max()}   "
      f"z {q[:,2].min()}-{q[:,2].max()}   of 0-{NUM_BINS-1}")

## Figure

Walls: the full 128-bin lattice, projected onto three faces of the quantization box.
Inside: the *actual* 3D lattice, at full resolution, in a `LOCAL_N`-bin cube around every
tokenized vertex -- it passes through the (semi-transparent) mesh, and each orange vertex
sits on one of its intersections. A full-volume lattice would be 3 x 128^2 = 49k lines and
render as solid grey, so it is only drawn where tokens actually live.

In [ ]:
def segments(pairs):
    """[(p, q), ...] -> x, y, z lists with None breaks, for one Scatter3d line trace."""
    xs, ys, zs = [], [], []
    for a, b in pairs:
        xs += [a[0], b[0], None]; ys += [a[1], b[1], None]; zs += [a[2], b[2], None]
    return xs, ys, zs


def lines(pairs, name, color, width, dash=None, **kw):
    x, y, z = segments(pairs)
    return go.Scatter3d(x=x, y=y, z=z, mode="lines", name=name, hoverinfo="skip",
                        line=dict(color=color, width=width, dash=dash), **kw)


def box_edges(lo, hi):
    c = np.array([[x, y, z] for x in (lo[0], hi[0]) for y in (lo[1], hi[1]) for z in (lo[2], hi[2])])
    return [(c[i], c[j]) for i in range(8) for j in range(i + 1, 8)
            if np.count_nonzero(c[i] != c[j]) == 1]


def plane_grid(plane):
    """Lattice lines on one face of the quantization box, split into (minor, major)."""
    axis, side = "xyz".index(plane[0]), plane[2:]
    # Nudged a hair inside the box: lines exactly coplanar with each other or with the
    # box edges z-fight in WebGL and render as broken dashes.
    eps = 2e-3 * scale.max()
    fixed = box_lo[axis] + eps if side == "min" else box_hi[axis] - eps
    a, b = [k for k in range(3) if k != axis]
    ticks = [box_lo[k] + np.arange(NUM_BINS) * step[k] for k in range(3)]
    minor, major = [], []
    for along, across in ((a, b), (b, a)):        # lines parallel to `across`
        for i, t in enumerate(ticks[along]):
            p, r = np.empty(3), np.empty(3)
            p[axis] = r[axis] = fixed
            p[along] = r[along] = t
            p[across], r[across] = box_lo[across], box_hi[across]
            is_major = MAJOR_EVERY and (i % MAJOR_EVERY == 0 or i == NUM_BINS - 1)
            if is_major:                          # majors one more hair inward, in front
                p[axis] = r[axis] = fixed + (eps if side == "min" else -eps)
            (major if is_major else minor).append((p, r))
    return minor, major


def local_lattice(centers, n):
    """Full-resolution 3D lattice in an n-bin cube around each lattice point in `centers`.

    The cube spans [c - n/2, c + n/2] bins per axis, clipped to the quantization box; lines
    sit at every integer bin inside it and run the full width of the cube. Returns the
    segments (in metres) and the unique lattice points they pass through.
    """
    h = n / 2
    pairs, nodes = [], set()
    for c in np.unique(centers, axis=0):
        lo = np.maximum(c - h, 0)
        hi = np.minimum(c + h, NUM_BINS - 1)
        ticks = [np.arange(np.ceil(lo[k]), np.floor(hi[k]) + 1) for k in range(3)]
        for ax in range(3):                       # lines parallel to `ax`
            a, b = [k for k in range(3) if k != ax]
            for ta in ticks[a]:
                for tb in ticks[b]:
                    p, r = np.empty(3), np.empty(3)
                    p[a] = r[a] = ta; p[b] = r[b] = tb
                    p[ax], r[ax] = lo[ax], hi[ax]
                    pairs.append((p, r))
        nodes.update(map(tuple, np.stack(np.meshgrid(*ticks, indexing="ij"), -1).reshape(-1, 3)))
    to_metres = lambda b: to_m(dequantize(b, NUM_BINS))
    return [(to_metres(p), to_metres(r)) for p, r in pairs], to_metres(np.array(sorted(nodes)))


minor, major = [], []
for plane in GRID_PLANES:
    mi, ma = plane_grid(plane)
    minor += mi; major += ma

local_pairs, local_nodes = local_lattice(q, LOCAL_N)
print(f"local lattice: {len(local_pairs)} segments, {len(local_nodes)} lattice points")

mesh_edges = {tuple(sorted(e)) for t in f2 for e in ((t[0], t[1]), (t[1], t[2]), (t[2], t[0]))}
qmesh_edges = {tuple(sorted(e)) for t in qf for e in ((t[0], t[1]), (t[1], t[2]), (t[2], t[0]))}
Qv_m = to_m(dequantize(qv, NUM_BINS))

traces = [
    lines(minor, f"lattice ({NUM_BINS} bins/axis)", COLORS["grid_minor"], 1),
    lines(major, f"every {MAJOR_EVERY} bins", COLORS["grid_major"], 1.5),
    lines(box_edges(box_lo, box_hi), "quantization box", COLORS["qbox"], 3),
    lines(box_edges(l1_lo, l1_hi), "LOD1 box (frame reference)", COLORS["lod1box"], 2.5),
    go.Mesh3d(x=V_m[:, 0], y=V_m[:, 1], z=V_m[:, 2], i=f2[:, 0], j=f2[:, 1], k=f2[:, 2],
              name="LOD2 surface", showlegend=True, color=COLORS["mesh"], flatshading=True,
              opacity=MESH_OPACITY,
              lighting=dict(ambient=0.55, diffuse=0.8, specular=0.05, roughness=0.9),
              lightposition=dict(x=1e5, y=-5e4, z=1e5), hoverinfo="skip"),
    lines([(V_m[a], V_m[b]) for a, b in mesh_edges], "LOD2 edges", COLORS["edges"], 3),
    lines(local_pairs, f"3D lattice, {LOCAL_N}^3 bins per vertex", COLORS["local"], 1),
    go.Scatter3d(x=local_nodes[:, 0], y=local_nodes[:, 1], z=local_nodes[:, 2], mode="markers",
                 name="lattice points", marker=dict(size=1.5, color=COLORS["local"]),
                 hoverinfo="skip", visible="legendonly"),
    lines([(Qv_m[a], Qv_m[b]) for a, b in qmesh_edges], "quantized edges", COLORS["snapped"], 3,
          visible="legendonly"),
    go.Scatter3d(x=V_m[:, 0], y=V_m[:, 1], z=V_m[:, 2], mode="markers", name="vertices",
                 marker=dict(size=2.5, color=COLORS["vert"]), visible="legendonly",
                 customdata=err * 100,
                 hovertemplate="(%{x:.2f}, %{y:.2f}, %{z:.2f}) m<br>snap %{customdata:.1f} cm<extra></extra>"),
    go.Scatter3d(x=Vq_m[:, 0], y=Vq_m[:, 1], z=Vq_m[:, 2], mode="markers",
                 name="snapped to lattice", marker=dict(size=5, color=COLORS["snapped"],
                 line=dict(color="white", width=0.5)), customdata=q,
                 hovertemplate="tokens (%{customdata[0]}, %{customdata[1]}, %{customdata[2]})<extra></extra>"),
]

hidden = dict(visible=False, showbackground=False, showgrid=False, zeroline=False)
fig = go.Figure(traces)
fig.update_layout(
    template="plotly_white", paper_bgcolor="white", plot_bgcolor="white", height=HEIGHT,
    margin=dict(l=0, r=0, t=50, b=0),
    title=dict(x=0.5, font=dict(size=15), text=(
        f"{NUM_BINS}-bin tokenization lattice  |  bin {step[0]*100:.1f} x {step[1]*100:.1f}"
        f" x {step[2]*100:.1f} cm  |  {len(f2)} triangles")),
    legend=dict(x=0.01, y=0.98, bgcolor="rgba(255,255,255,0.85)", bordercolor="#d0d3d8",
                borderwidth=1, font=dict(size=12)),
    scene=dict(xaxis=hidden, yaxis=hidden, zaxis=hidden, aspectmode="data", camera=CAMERA),
)
EXPORT = dict(toImageButtonOptions=dict(format="png", filename="tokenization_grid", scale=3))
fig.show(config=EXPORT)

## Canonical vertex order

The order `canonicalize` gives the *quantized* vertices before faces are written out:
duplicates merged, then sorted ascending by bin index with **z primary, then y, then x**
(`np.lexsort((x, y, z))`). The label on each vertex is its index 0 .. m-1 in that order --
the index faces refer to, so it also decides face order and therefore token order.
Vertices on one z level (e.g. the whole ground ring) are ordered by y, then x.

Toggle "order path" in the legend to see the sweep as one polyline.

In [ ]:
LABEL_SIZE = 11
ORDER_SCALE = [[0.0, "#2d1e5c"], [0.5, "#b8327a"], [1.0, "#f28c28"]]  # dark -> orange, legible on white

m = len(qv)
order = np.arange(m)
# canonicalize's contract, checked rather than assumed: rows ascend lexicographically in (z, y, x).
keys = qv[:, ::-1]
assert all(tuple(keys[i]) < tuple(keys[i + 1]) for i in range(m - 1)), "not in (z, y, x) order"

order_fig = go.Figure([
    lines(box_edges(box_lo, box_hi), "quantization box", COLORS["qbox"], 2),
    go.Mesh3d(x=Qv_m[:, 0], y=Qv_m[:, 1], z=Qv_m[:, 2], i=qf[:, 0], j=qf[:, 1], k=qf[:, 2],
              name="LOD2 surface (quantized)", showlegend=True, color=COLORS["mesh"],
              flatshading=True, opacity=MESH_OPACITY,
              lighting=dict(ambient=0.55, diffuse=0.8, specular=0.05, roughness=0.9),
              lightposition=dict(x=1e5, y=-5e4, z=1e5), hoverinfo="skip"),
    lines([(Qv_m[a], Qv_m[b]) for a, b in qmesh_edges], "edges", COLORS["edges"], 2),
    go.Scatter3d(x=Qv_m[:, 0], y=Qv_m[:, 1], z=Qv_m[:, 2], mode="lines", name="order path",
                 line=dict(color=order, colorscale=ORDER_SCALE, width=4), hoverinfo="skip",
                 visible="legendonly"),
    go.Scatter3d(x=Qv_m[:, 0], y=Qv_m[:, 1], z=Qv_m[:, 2], mode="markers+text",
                 name="vertices (canonical index)", text=[str(i) for i in order],
                 textposition="top center", textfont=dict(size=LABEL_SIZE, color="#1b1b1b"),
                 marker=dict(size=5, color=order, colorscale=ORDER_SCALE,
                             line=dict(color="white", width=0.5),
                             colorbar=dict(title="index", thickness=14, len=0.5, x=0.98)),
                 customdata=np.column_stack([order, qv]),
                 hovertemplate=("vertex %{customdata[0]}<br>"
                                "bins x %{customdata[1]}, y %{customdata[2]}, z %{customdata[3]}"
                                "<extra></extra>")),
])
order_fig.update_layout(
    template="plotly_white", paper_bgcolor="white", plot_bgcolor="white", height=HEIGHT,
    margin=dict(l=0, r=0, t=50, b=0),
    title=dict(x=0.5, font=dict(size=15),
               text=f"Canonical vertex order (z, then y, then x)  |  {m} vertices, 0-{m - 1}"),
    legend=dict(x=0.01, y=0.98, bgcolor="rgba(255,255,255,0.85)", bordercolor="#d0d3d8",
                borderwidth=1, font=dict(size=12)),
    scene=dict(xaxis=hidden, yaxis=hidden, zaxis=hidden, aspectmode="data", camera=CAMERA),
)
order_fig.show(config=dict(toImageButtonOptions=dict(format="png", filename="canonical_order", scale=3)))

## Canonical face order

After the vertex sort, each face is rotated cyclically so its lowest vertex index comes
first (rotation, not re-sorting, so the winding -- and the outward normal -- survives), then
faces are sorted lexicographically by their index triple. Degenerate and duplicate faces are
dropped along the way.

Each triangle is coloured by and labelled with its face index `f` at its centroid; its three
corners carry `(f, k)`, `k` = position within the face, i.e. the order the face's three
vertices are written into the token stream. `(f,0)` is always the face's lowest-index vertex,
and `(f,0) -> (f,1) -> (f,2)` keeps the source winding -- counter-clockwise seen from
outside when the source is consistently wound (true for the default building: `fix_winding`
flips 0 of its 140 faces; not guaranteed for every LOD2 object).
Corner labels are pulled `CORNER_INSET` of the way toward the centroid so the labels of
faces sharing a vertex don't stack. Hover a corner for its vertex index.

With 140 faces it is busy: rotate, toggle traces in the legend, or set `FACE_SUBSET` to a
range (e.g. `range(0, 20)`) to label only those faces.

In [ ]:
FACE_SUBSET = None        # None = label every face; or e.g. range(0, 20)
CORNER_INSET = 0.3        # 0 = on the vertex, 1 = on the centroid
LABEL_LIFT = 0.3          # metres off the surface along the outward normal, so labels are not cut by it
                          # (camera-facing text still clips walls seen edge-on; raise it if so)
FACE_OPACITY = 1.0        # opaque: labels are depth-tested, so back faces stay hidden until rotated to
# Light version of ORDER_SCALE so dark labels stay legible on every face.
FACE_SCALE = [[0.0, "#cfc8e6"], [0.5, "#f0c4da"], [1.0, "#fcd9b0"]]
FACE_LABEL_SIZE, CORNER_LABEL_SIZE = 12, 9

F = len(qf)
# canonicalize's face contract: each face leads with its lowest index, faces ascend lexicographically.
assert (qf[:, 0] == qf.min(axis=1)).all(), "face not rotated to its lowest index"
assert all(tuple(qf[i]) < tuple(qf[i + 1]) for i in range(F - 1)), "faces not sorted"

shown = np.arange(F) if FACE_SUBSET is None else np.asarray(list(FACE_SUBSET))
tri = Qv_m[qf]                                    # [F, 3, 3] corner positions, metres
cent = tri.mean(axis=1)                           # [F, 3]
corner_pos = tri + CORNER_INSET * (cent[:, None, :] - tri)

# Outward normals for the lift only. fix_winding reverses rows in place (row i is still face i),
# so it orients the source's inconsistently wound faces without touching the canonical corner order.
out = Qv_m[fix_winding(Qv_m, qf)]
nrm = np.cross(out[:, 1] - out[:, 0], out[:, 2] - out[:, 0])
nrm /= np.maximum(np.linalg.norm(nrm, axis=1, keepdims=True), 1e-12)
cent = cent + LABEL_LIFT * nrm
corner_pos = corner_pos + LABEL_LIFT * nrm[:, None, :]

fs, ks = np.meshgrid(shown, np.arange(3), indexing="ij")    # every shown (face, corner) pair
fs, ks = fs.ravel(), ks.ravel()
cp = corner_pos[fs, ks]

face_fig = go.Figure([
    go.Mesh3d(x=Qv_m[:, 0], y=Qv_m[:, 1], z=Qv_m[:, 2], i=qf[:, 0], j=qf[:, 1], k=qf[:, 2],
              intensity=np.arange(F), intensitymode="cell", colorscale=FACE_SCALE,
              colorbar=dict(title="face", thickness=14, len=0.5, x=0.98),
              name="faces (by index)", showlegend=True, flatshading=True, opacity=FACE_OPACITY,
              lighting=dict(ambient=0.9, diffuse=0.25, specular=0.0, roughness=1.0),
              hoverinfo="skip"),
    lines([(Qv_m[a], Qv_m[b]) for a, b in qmesh_edges], "edges", "#5a6a80", 2),
    go.Scatter3d(x=cent[shown, 0], y=cent[shown, 1], z=cent[shown, 2], mode="text",
                 name="face index f", text=[str(f) for f in shown],
                 textfont=dict(size=FACE_LABEL_SIZE, color="#111111"),
                 customdata=np.column_stack([shown, qf[shown]]),
                 hovertemplate=("face %{customdata[0]}: vertices "
                                "(%{customdata[1]}, %{customdata[2]}, %{customdata[3]})<extra></extra>")),
    go.Scatter3d(x=cp[:, 0], y=cp[:, 1], z=cp[:, 2], mode="text", name="corner (f, k)",
                 text=[f"({f},{k})" for f, k in zip(fs, ks)],
                 textfont=dict(size=CORNER_LABEL_SIZE, color="#3b4250"),
                 customdata=np.column_stack([fs, ks, qf[fs, ks]]),
                 hovertemplate=("face %{customdata[0]}, corner %{customdata[1]}"
                                "<br>vertex %{customdata[2]}<extra></extra>")),
])
face_fig.update_layout(
    template="plotly_white", paper_bgcolor="white", plot_bgcolor="white", height=HEIGHT,
    margin=dict(l=0, r=0, t=50, b=0),
    title=dict(x=0.5, font=dict(size=15),
               text=f"Canonical face order  |  {F} faces, 0-{F - 1}  |  corners (f, k)"),
    legend=dict(x=0.01, y=0.98, bgcolor="rgba(255,255,255,0.85)", bordercolor="#d0d3d8",
                borderwidth=1, font=dict(size=12)),
    scene=dict(xaxis=hidden, yaxis=hidden, zaxis=hidden, aspectmode="data", camera=CAMERA),
)
face_fig.show(config=dict(toImageButtonOptions=dict(format="png", filename="canonical_face_order", scale=3)))

## Zoom: one vertex in its lattice cell

The worst-snapped roof vertex with its neighbourhood of the 3D lattice. Black is the
source coordinate, orange the lattice point it is tokenized as; the orange box is that
point's rounding cell (half a bin each way) -- every coordinate inside it gets the same
three tokens.

In [ ]:
R = 1                                            # neighbourhood half-width, in bins
roof = np.flatnonzero(V_m[:, 2] > V_m[:, 2].min() + 1.0)   # skip ground-plane vertices
w = int(roof[np.argmax(err[roof])])
c = q[w]
ks = [np.arange(c[k] - R, c[k] + R + 1) for k in range(3)]
pt = lambda i, j, k: to_m(dequantize(np.array([i, j, k]), NUM_BINS))

lattice = []
for j in ks[1]:
    for k in ks[2]:
        lattice.append((pt(ks[0][0], j, k), pt(ks[0][-1], j, k)))
for i in ks[0]:
    for k in ks[2]:
        lattice.append((pt(i, ks[1][0], k), pt(i, ks[1][-1], k)))
for i in ks[0]:
    for j in ks[1]:
        lattice.append((pt(i, j, ks[2][0]), pt(i, j, ks[2][-1])))

nodes = np.array([pt(i, j, k) for i in ks[0] for j in ks[1] for k in ks[2]])
inc = f2[(f2 == w).any(axis=1)]                  # triangles touching this vertex
lo_z, hi_z = pt(*(c - R - 0.5)), pt(*(c + R + 0.5))
cell = box_edges(pt(*(c - 0.5)), pt(*(c + 0.5)))

zoom = go.Figure([
    lines(lattice, "lattice", COLORS["grid_major"], 2),
    go.Scatter3d(x=nodes[:, 0], y=nodes[:, 1], z=nodes[:, 2], mode="markers", name="lattice points",
                 marker=dict(size=2, color=COLORS["grid_major"]), hoverinfo="skip"),
    go.Mesh3d(x=V_m[:, 0], y=V_m[:, 1], z=V_m[:, 2], i=inc[:, 0], j=inc[:, 1], k=inc[:, 2],
              color=COLORS["mesh"], opacity=0.5, name="incident triangles", showlegend=True,
              hoverinfo="skip"),
    lines(cell, "rounding cell", COLORS["snapped"], 2, dash="dash"),
    lines([(V_m[w], Vq_m[w])], f"snap {err[w]*100:.1f} cm", COLORS["vert"], 4),
    go.Scatter3d(x=[V_m[w, 0]], y=[V_m[w, 1]], z=[V_m[w, 2]], mode="markers", name="source vertex",
                 marker=dict(size=6, color=COLORS["vert"])),
    go.Scatter3d(x=[Vq_m[w, 0]], y=[Vq_m[w, 1]], z=[Vq_m[w, 2]], mode="markers",
                 name=f"tokens ({c[0]}, {c[1]}, {c[2]})",
                 marker=dict(size=7, color=COLORS["snapped"], line=dict(color="white", width=1))),
])
axis = lambda k, title: dict(title=title, range=[lo_z[k], hi_z[k]], backgroundcolor="white",
                             showbackground=False, gridcolor="#eef0f3", zeroline=False)
zoom.update_layout(
    template="plotly_white", paper_bgcolor="white", height=650, margin=dict(l=0, r=0, t=40, b=0),
    title=dict(x=0.5, text=f"vertex {w}: {R*2+1}^3 lattice neighbourhood"),
    legend=dict(x=0.01, y=0.98, bgcolor="rgba(255,255,255,0.85)"),
    scene=dict(xaxis=axis(0, "x [m]"), yaxis=axis(1, "y [m]"), zaxis=axis(2, "z [m]"),
               aspectmode="data", camera=CAMERA),
)
zoom.show(config=dict(toImageButtonOptions=dict(format="png", filename="tokenization_grid_zoom", scale=3)))

## Face copying rate, LOD1 -> LOD2

A copied vertex means little; what lets the model echo its input is a whole **face** that is
identical in condition and target. Here a face is its 9-token row after `canonicalize`, taken
from `MeshDataset._tokens` -- the exact sequences the model trains on (coord tokenization,
LOD1 frame, margins, `max_faces`). A target face is **copied** if the same 9-token row occurs
anywhere in the condition.

Matching rows across the two meshes is sound because the rotation `canonicalize` applies
(lowest vertex index first) is decided by the (z, y, x) order of the face's own three grid
points, which is the same in either mesh -- asserted below for every face, not assumed.

Not copied by this definition, on purpose: the same surface triangulated differently, and the
same triangle with opposite winding (a different token sequence; counted separately as
`flipped` because it shows how much copying source orientation hides).

Reported:
- **micro** = copied target faces / all target faces -- the share of the training signal that
  is verbatim in the condition; **macro** = mean per-building rate;
- buildings whose target is the condition token-for-token, and at >= 50 / >= 90 / 0 %;
- copy rate per face class (ground / wall / roof, from the quantized geometry), by position in
  the target sequence (canonical order runs bottom-up), and by target size.

Population is the whole filtered dataset, before the train/val/test split.
`mini_cleaner` + real `LOD1` matches the v6 runs; `mini_remote` + `LOD1_synth` the v2-v5 ones.
`cleaner` is the v6 training set itself but takes a while to load.

In [ ]:
import json
import logging
import subprocess

import pandas as pd
from plotly.subplots import make_subplots

from src.dataset.mesh_dataset import MeshDataset

STATS_DATASET   = ROOT / "data" / "The Hague" / "mini_cleaner"
STATS_LOD_IN    = "LOD1"            # "LOD1_synth" for the v2-v5 setup (with mini_remote)
STATS_MAX_FILES = None              # e.g. 10 for a quick look
SAVE_DIR        = ROOT / "outputs" / "face_copy"   # None to skip writing the per-building CSV

logging.basicConfig(level=logging.WARNING)        # MeshDataset logs margins at INFO
ds = MeshDataset(STATS_DATASET, lod_in=STATS_LOD_IN, lod_out="LOD2", num_bins=NUM_BINS,
                 margin_lo=MARGIN_LO, margin_hi=MARGIN_HI, max_faces=cfg.max_faces,
                 max_files=STATS_MAX_FILES, tokenization="coord")
print(f"{STATS_DATASET.name} / {STATS_LOD_IN} -> LOD2: {len(ds)} pairs (max_faces={cfg.max_faces})")

In [ ]:
FLIP = [0, 1, 2, 6, 7, 8, 3, 4, 5]   # (a, b, c) -> (a, c, b): same triangle, reversed, still a-first
CLASSES = ("ground", "wall", "roof")
POS_BINS = 10
WALL_NZ = 0.15                         # |n_z| below this is a wall


def check_rotation(rows):
    """Every face must lead with its (z, y, x)-smallest grid point -- the cross-mesh invariant."""
    v = rows.reshape(-1, 3, 3)
    key = (v[..., 2] * NUM_BINS + v[..., 1]) * NUM_BINS + v[..., 0]
    assert (key[:, :1] < key[:, 1:]).all(), "face rotation is not geometry-determined"


def face_class(rows, scale):
    """0 ground / 1 wall / 2 roof for each face, from its quantized geometry in metres.

    Ground = every corner on the mesh's lowest z bin. Wall = near-vertical. Roof = the rest
    (flat and pitched). Degenerate zero-area rows land in roof; they are rare after canonicalize.
    """
    v = rows.reshape(-1, 3, 3)
    z = v[..., 2]
    ground = (z == z.min()).all(axis=1)
    tri = (v / (NUM_BINS - 1) - 0.5) * scale
    n = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    nz = np.abs(n[:, 2]) / np.maximum(np.linalg.norm(n, axis=1), 1e-12)
    return np.where(ground, 0, np.where(nz < WALL_NZ, 1, 2))


records = []
pos_hit, pos_tot = np.zeros(POS_BINS), np.zeros(POS_BINS)
for i in range(len(ds)):
    item = ds._tokens(i)
    c = item["cond"].numpy().reshape(-1, 9)
    t = item["tgt"].numpy()[1:-1].reshape(-1, 9)            # strip BOS / EOS
    if len(t) == 0:
        continue
    check_rotation(c); check_rotation(t)

    cond = set(map(tuple, c))
    hit = np.fromiter((tuple(r) in cond for r in t), bool, len(t))
    flip = ~hit & np.fromiter((tuple(r) in cond for r in t[:, FLIP]), bool, len(t))
    cls = face_class(t, item["scale"].numpy())

    b = np.minimum(np.arange(len(t)) * POS_BINS // len(t), POS_BINS - 1)
    np.add.at(pos_hit, b, hit); np.add.at(pos_tot, b, 1)

    rec = dict(id=item["id"], file=ds.files[i], n_cond=len(c), n_tgt=len(t),
               copied=int(hit.sum()), flipped=int(flip.sum()),
               seq_identical=bool(len(c) == len(t) and (c == t).all()))
    for k, name in enumerate(CLASSES):
        rec[f"n_{name}"] = int((cls == k).sum())
        rec[f"copied_{name}"] = int(hit[cls == k].sum())
    records.append(rec)

df = pd.DataFrame(records)
df["frac"] = df.copied / df.n_tgt
print(f"{len(df)} buildings, {df.n_tgt.sum()} target faces; rotation invariant held on every face")

In [ ]:
tot = df[["n_tgt", "copied", "flipped"]].sum()
summary = pd.Series({
    "faces copied verbatim (micro)":                    tot.copied / tot.n_tgt,
    "mean per-building copy rate (macro)":              df.frac.mean(),
    "median per-building copy rate":                    df.frac.median(),
    "buildings: target == condition, token for token":  df.seq_identical.mean(),
    "buildings: 100% of target faces copied":           (df.copied == df.n_tgt).mean(),
    "buildings: >= 90% copied":                         (df.frac >= 0.9).mean(),
    "buildings: >= 50% copied":                         (df.frac >= 0.5).mean(),
    "buildings: no face copied":                        (df.copied == 0).mean(),
    "faces matching only with flipped winding":         tot.flipped / tot.n_tgt,
}, name=f"{STATS_DATASET.name} / {STATS_LOD_IN}")
display(summary.map("{:.1%}".format).to_frame())

per_class = pd.DataFrame({
    "share of target faces": [df[f"n_{k}"].sum() / tot.n_tgt for k in CLASSES],
    "copy rate":             [df[f"copied_{k}"].sum() / max(df[f"n_{k}"].sum(), 1) for k in CLASSES],
    "share of copied faces": [df[f"copied_{k}"].sum() / max(tot.copied, 1) for k in CLASSES],
}, index=CLASSES)
display(per_class.style.format("{:.1%}"))

In [ ]:
SIZE_EDGES = [0, 12, 24, 50, 100, 200]      # target triangles
size_bin = pd.cut(df.n_tgt, SIZE_EDGES, labels=[f"{a + 1}-{b}" for a, b in zip(SIZE_EDGES, SIZE_EDGES[1:])])
by_size = df.groupby(size_bin, observed=False).agg(copied=("copied", "sum"), n=("n_tgt", "sum"),
                                                   buildings=("id", "size"))

stats_fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.08, subplot_titles=(
    "per-building copy rate", "copy rate by position in target sequence",
    "copy rate by target size (faces)"))
stats_fig.add_bar(x=np.linspace(0.025, 0.975, 20), y=np.histogram(df.frac, bins=20, range=(0, 1))[0],
                  width=0.045, marker_color=COLORS["edges"], row=1, col=1,
                  hovertemplate="%{x:.0%}: %{y} buildings<extra></extra>")
stats_fig.add_bar(x=[f"{10 * k}-{10 * k + 10}%" for k in range(POS_BINS)], y=pos_hit / pos_tot,
                  marker_color=COLORS["edges"], row=1, col=2,
                  hovertemplate="%{x}: %{y:.1%}<extra></extra>")
stats_fig.add_bar(x=by_size.index.astype(str), y=by_size.copied / by_size.n, customdata=by_size.buildings,
                  marker_color=COLORS["edges"], row=1, col=3,
                  hovertemplate="%{x} faces: %{y:.1%} (%{customdata} buildings)<extra></extra>")
stats_fig.update_yaxes(title_text="buildings", row=1, col=1)
stats_fig.update_yaxes(tickformat=".0%", range=[0, 1], row=1, col=2)
stats_fig.update_yaxes(tickformat=".0%", range=[0, 1], row=1, col=3)
stats_fig.update_xaxes(tickformat=".0%", row=1, col=1)
stats_fig.update_layout(template="plotly_white", paper_bgcolor="white", height=420, showlegend=False,
                        margin=dict(l=40, r=20, t=100, b=40),
                        title=dict(x=0.5, y=0.97, text=f"Face copying, {STATS_DATASET.name} / {STATS_LOD_IN} -> LOD2"))
stats_fig.show(config=dict(toImageButtonOptions=dict(format="png", filename="face_copy_rate", scale=3)))

In [ ]:
if SAVE_DIR is not None:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    stem = f"{STATS_DATASET.name}_{STATS_LOD_IN}" + (f"_files{STATS_MAX_FILES}" if STATS_MAX_FILES else "")
    df.to_csv(SAVE_DIR / f"{stem}.csv", index=False)
    try:
        commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True,
                                text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        commit = None
    meta = dict(dataset=str(STATS_DATASET.relative_to(ROOT)), lod_in=STATS_LOD_IN, lod_out="LOD2",
                num_bins=NUM_BINS, margin_lo=list(MARGIN_LO), margin_hi=list(MARGIN_HI),
                max_faces=cfg.max_faces, max_files=STATS_MAX_FILES, tokenization="coord",
                git_commit=commit, summary=summary.to_dict(), per_class=per_class.to_dict())
    (SAVE_DIR / f"{stem}.json").write_text(json.dumps(meta, indent=2))
    print(f"wrote {SAVE_DIR / stem}.csv / .json")